# Exploración del dataset generado con Spark pipeline

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

df = spark.read.parquet("../../data/features/m5_features")

print("Rows:", df.count())
print("Columns:", len(df.columns))

In [ ]:
df.printSchema()

In [ ]:
df.groupBy("store_id").count().show()

In [ ]:
import pandas as pd

df = pd.read_parquet(
    "../../data/features/m5_features"
)

df["store_id"].unique()

## Estrategia de entrenamiento por subconjuntos jerárquicos

El dataset M5, tras las transformaciones realizadas en la fase ETL mediante Apache Spark, contiene aproximadamente 58 millones de registros. Este volumen de datos, aunque manejable en operaciones distribuidas, puede resultar computacionalmente costoso durante la fase de entrenamiento de modelos en entornos locales basados en pandas.

Por este motivo, se adopta una estrategia de entrenamiento progresivo basada en subconjuntos jerárquicos, concretamente por identificador de tienda (`store_id`). Este enfoque permite mantener la coherencia estructural del dataset y preservar la naturaleza jerárquica del problema.

La estrategia consiste en:

- Entrenar modelos utilizando inicialmente una tienda individual.
- Repetir el mismo pipeline de entrenamiento para otras tiendas.
- Comparar posteriormente el rendimiento entre subconjuntos.
- Evaluar la escalabilidad del enfoque para múltiples niveles jerárquicos.

Ejemplo conceptual:

- CA_1 → entrenamiento de modelo
- CA_2 → entrenamiento de modelo
- TX_1 → entrenamiento de modelo
- WI_1 → entrenamiento de modelo

Este enfoque presenta varias ventajas técnicas:

- Reduce el consumo de memoria durante el entrenamiento.
- Facilita el proceso de depuración y validación del pipeline.
- Permite escalar progresivamente el sistema.
- Mantiene la validez metodológica en problemas jerárquicos.

En esta fase inicial se seleccionará una tienda representativa (`CA_1`) para desarrollar el primer modelo base.

## Selección inicial de subconjunto de entrenamiento

Como primer paso en la fase de modelado, se selecciona un subconjunto del dataset correspondiente a una única tienda (`store_id = CA_1`).

Esta selección permite:

- Reducir el volumen de datos procesados en memoria.
- Validar el comportamiento del modelo en un entorno controlado.
- Establecer un pipeline reproducible que posteriormente podrá aplicarse al resto de tiendas.

Una vez validado el modelo base sobre este subconjunto, el mismo procedimiento podrá repetirse para otras tiendas, permitiendo una evaluación comparativa entre distintos niveles jerárquicos.

## Verificación del rango temporal del subconjunto

Tras seleccionar el subconjunto correspondiente a una tienda específica (`store_id`), se verifica el rango temporal disponible en los datos.

Este paso permite confirmar que el conjunto contiene la serie temporal completa y que el proceso de transformación previo no ha eliminado observaciones relevantes.

La verificación del rango temporal constituye un paso fundamental antes de aplicar la división temporal en conjuntos de entrenamiento, validación y prueba.

In [ ]:
store = "CA_1"

df_store = df[
    df["store_id"] == store
].copy()

print(df_store.shape)

print(df_store["date"].min())
print(df_store["date"].max())
print(df_store["d"].max())

In [ ]:
df_store.isna().sum().sort_values(ascending=False)

## División temporal del conjunto de datos

Dado que el problema corresponde a series temporales, la división del dataset no se realiza de forma aleatoria.

En su lugar, se establece una separación estrictamente temporal entre los conjuntos de entrenamiento, validación y prueba, garantizando que los datos futuros no influyan en el aprendizaje del modelo.

Este enfoque evita la aparición de data leakage y permite evaluar el rendimiento del modelo en condiciones realistas.

In [ ]:
df_store["date"] = pd.to_datetime(df_store["date"])

train = df_store[
    df_store["date"] < "2015-01-01"
]

validation = df_store[
    (df_store["date"] >= "2015-01-01") &
    (df_store["date"] < "2016-01-01")
]

test = df_store[
    df_store["date"] >= "2016-01-01"
]

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

In [ ]:
df_store.head()

In [ ]:
df_store.tail()

## Preparación de variables predictoras y variable objetivo

Una vez realizada la división temporal del dataset, se procede a separar las variables predictoras (`features`) y la variable objetivo (`target`).

La variable objetivo corresponde a la demanda diaria (`sales`), mientras que el resto de variables derivadas durante la fase de ingeniería de características se emplean como variables explicativas.

Este procedimiento permite preparar los datos en el formato requerido por los algoritmos de aprendizaje supervisado basados en árboles.

In [ ]:
# preparar variables 
target = "sales"

features = [
    col for col in df_store.columns
    if col not in [
        "sales",
        "date",
        "id",
        "d"
    ]
]

X_train = train[features].copy()
y_train = train[target].copy()

X_val = validation[features].copy()
y_val = validation[target].copy()

X_test = test[features].copy()
y_test = test[target].copy()

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

In [ ]:
# revisar tipos porque LightGBM no acepta objects
X_train.dtypes.value_counts()

In [ ]:
X_train.select_dtypes(include="object").columns

In [ ]:
# Re-crear datasets correctamente

X_train = train[features].copy()
y_train = train[target].copy()

X_val = validation[features].copy()
y_val = validation[target].copy()

X_test = test[features].copy()
y_test = test[target].copy()


# Convertir categóricas correctamente

categorical_cols = [
    "store_id",
    "item_id",
    "dept_id",
    "cat_id",
    "state_id",
    "weekday",
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2"
]

for col in categorical_cols:

    X_train[col] = X_train[col].astype("category")
    X_val[col] = X_val[col].astype("category")
    X_test[col] = X_test[col].astype("category")

In [ ]:
X_train.select_dtypes(include="object").columns

## Entrenamiento del modelo LightGBM

Se selecciona LightGBM como modelo principal debido a su eficiencia computacional y su alto rendimiento en problemas de forecasting con datos tabulares de alta dimensionalidad.

Este algoritmo permite manejar grandes volúmenes de datos y variables categóricas de forma eficiente, lo que lo convierte en una opción adecuada para el dataset M5.

El modelo se entrena utilizando el conjunto de entrenamiento y se valida posteriormente sobre el conjunto de validación temporal.

In [ ]:
# Entrenar LightGBM
import lightgbm as lgb

model = lgb.LGBMRegressor(
    n_estimators=100,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    categorical_feature=categorical_cols
)

## Interpretación de los mensajes de entrenamiento del modelo

Durante el proceso de entrenamiento del modelo LightGBM se generan diversos mensajes informativos relacionados con la estructura de los datos y el uso de recursos computacionales.

Uno de los mensajes indica que algunas variables categóricas contienen un número elevado de categorías. Este comportamiento es esperado en el dataset M5, especialmente en variables como `item_id`, que pueden contener miles de valores distintos. Esta advertencia no representa un error, sino una característica inherente del problema.

Asimismo, el sistema informa que se ha seleccionado automáticamente el modo de ejecución paralelo ("row-wise multi-threading"). Este comportamiento confirma que el modelo está utilizando correctamente los recursos disponibles del procesador para acelerar el entrenamiento.

Finalmente, el mensaje que indica el número total de observaciones utilizadas en el entrenamiento confirma que el modelo ha sido entrenado sobre más de cuatro millones de registros, lo que valida la correcta preparación del conjunto de datos.

## Evaluación del modelo predictivo

Una vez entrenado el modelo LightGBM, se procede a evaluar su rendimiento utilizando el conjunto de prueba temporal.

Para cuantificar la precisión del modelo, se emplean métricas de error habituales en problemas de forecasting, como el Error Cuadrático Medio (RMSE) y el Error Absoluto Medio (MAE).

Estas métricas permiten evaluar la capacidad del modelo para generalizar sobre datos no vistos previamente.

In [ ]:
y_pred = model.predict(X_test)

print("Predictions generated:", len(y_pred))

In [ ]:
# Calcular metricas
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

rmse = np.sqrt(
    mean_squared_error(y_test, y_pred)
)

mae = mean_absolute_error(
    y_test,
    y_pred
)

print("RMSE:", rmse)
print("MAE:", mae)

## Evaluación del rendimiento del modelo

Tras generar las predicciones sobre el conjunto de prueba temporal, se calcularon métricas de error para evaluar el rendimiento del modelo.

El Error Cuadrático Medio (RMSE) permite medir la magnitud media del error penalizando especialmente desviaciones grandes, mientras que el Error Absoluto Medio (MAE) proporciona una medida más interpretable del error medio cometido por el modelo.

Los valores obtenidos indican que el modelo LightGBM es capaz de capturar patrones relevantes en la serie temporal, proporcionando predicciones consistentes y estables sobre datos no observados durante el entrenamiento.

Este modelo constituye el primer baseline funcional del sistema predictivo y servirá como referencia para futuras mejoras y comparaciones con otros algoritmos.

In [ ]:
# Feature importance 
import pandas as pd

importance = pd.DataFrame({
    "feature": features,
    "importance": model.feature_importances_
})

importance = importance.sort_values(
    by="importance",
    ascending=False
)

importance.head(15)

## Análisis de importancia de variables

Con el objetivo de comprender el comportamiento del modelo, se analizó la importancia relativa de las variables predictoras.

La importancia de variables permite identificar cuáles son los factores que contribuyen en mayor medida a las predicciones del modelo. Este análisis proporciona información valiosa sobre la relevancia de las variables temporales y contextuales en la predicción de la demanda.

Los resultados obtenidos servirán como base para posteriores análisis de interpretabilidad mediante técnicas más avanzadas, como SHAP.

## Interpretación de la importancia de variables

El análisis de importancia de variables revela que las variables temporales derivadas durante la fase de ingeniería de características tienen un papel fundamental en la predicción de la demanda.

En particular, las variables `rolling_mean_28` y `lag_7` aparecen entre las más influyentes, lo que indica que el comportamiento histórico reciente de la demanda constituye un factor determinante en la predicción futura.

Asimismo, variables categóricas como `item_id` presentan una elevada importancia, lo que sugiere que el modelo es capaz de diferenciar patrones específicos asociados a cada producto.

Las variables relacionadas con el calendario (`weekday`, `month`) y el precio (`sell_price`) también contribuyen significativamente al modelo, evidenciando la relevancia de factores temporales y comerciales en la dinámica de la demanda.

Este análisis confirma la validez de las variables generadas durante la fase ETL y proporciona evidencia empírica del correcto funcionamiento del modelo.

In [ ]:
# rafico Importance
import matplotlib.pyplot as plt

top_features = importance.head(15)

plt.figure()

plt.barh(
    top_features["feature"],
    top_features["importance"]
)

plt.gca().invert_yaxis()

plt.title("Top 15 Feature Importance")

plt.show()

## Visualización de la importancia de variables

Para facilitar la interpretación del modelo, se representa gráficamente la importancia relativa de las variables más influyentes.

Esta visualización permite identificar de forma intuitiva cuáles son los factores que contribuyen en mayor medida a las predicciones del modelo, facilitando la comunicación de resultados y el análisis posterior.

La representación gráfica constituye una herramienta clave en la interpretación de modelos basados en árboles.

## Entrenamiento del modelo Random Forest como baseline

Con el objetivo de establecer una referencia inicial de rendimiento, se entrena un modelo basado en Random Forest.

Este algoritmo se selecciona como baseline debido a su robustez frente a ruido y su capacidad para capturar relaciones no lineales sin requerir un ajuste complejo de hiperparámetros.

El modelo Random Forest permitirá comparar posteriormente su rendimiento con modelos más avanzados como LightGBM y XGBoost.

In [ ]:
# Random Forest no acepta variables categóricas tipo "category". Las convertimos a códigos numéricos
from sklearn.preprocessing import LabelEncoder

X_train_rf = X_train.copy()
X_val_rf = X_val.copy()
X_test_rf = X_test.copy()

encoders = {}

for col in categorical_cols:

    le = LabelEncoder()

    X_train_rf[col] = le.fit_transform(X_train_rf[col].astype(str))
    X_val_rf[col] = le.transform(X_val_rf[col].astype(str))
    X_test_rf[col] = le.transform(X_test_rf[col].astype(str))

    encoders[col] = le

In [ ]:
# Entrenar Random Forest
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=50,
    max_depth=10,
    n_jobs=-1,
    random_state=42
)

rf_model.fit(
    X_train_rf,
    y_train
)

## Entrenamiento del modelo Random Forest

El modelo Random Forest se entrena utilizando el conjunto de entrenamiento previamente definido.

Este modelo actúa como referencia inicial para evaluar el rendimiento de algoritmos más avanzados, permitiendo establecer una base comparativa sólida.

La capacidad del Random Forest para manejar múltiples variables predictoras lo convierte en una opción adecuada como baseline en problemas de forecasting.

In [ ]:
# redicciones Random Forest
y_pred_rf = rf_model.predict(X_test_rf)

In [ ]:
# Métricas Random Forest 
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

rmse_rf = np.sqrt(
    mean_squared_error(y_test, y_pred_rf)
)

mae_rf = mean_absolute_error(
    y_test,
    y_pred_rf
)

print("Random Forest RMSE:", rmse_rf)
print("Random Forest MAE:", mae_rf)

LightGBM
RMSE: 2.0777
MAE: 1.0412
Tiempo: 5–30 seg

Random Forest
RMSE: 2.0813
MAE: 1.0417
Tiempo: 7–8 min
Rendimiento ≈ igual
Pero LightGBM es muchísimo más rápido

## Comparación entre modelos Random Forest y LightGBM

Tras entrenar ambos modelos, se observa que Random Forest y LightGBM presentan valores de error muy similares en términos de RMSE y MAE.

No obstante, se identifica una diferencia significativa en el tiempo de entrenamiento, siendo LightGBM considerablemente más eficiente desde el punto de vista computacional.

Este resultado confirma la idoneidad de LightGBM como modelo principal del sistema, especialmente en escenarios con grandes volúmenes de datos, donde la eficiencia computacional constituye un factor crítico.

El modelo Random Forest cumple adecuadamente su función como baseline, proporcionando una referencia robusta frente a la cual comparar modelos más avanzados.

## Entrenamiento del modelo XGBoost

Como parte del análisis comparativo entre modelos basados en árboles, se entrena un modelo XGBoost.

Este algoritmo se basa en técnicas de gradient boosting y permite optimizar el proceso de aprendizaje mediante la construcción secuencial de árboles, mejorando progresivamente el rendimiento predictivo.

El modelo XGBoost se evaluará utilizando el mismo esquema temporal definido previamente, permitiendo una comparación directa con Random Forest y LightGBM.

In [ ]:
# Entrenar XGBoost
import xgboost as xgb

xgb_model = xgb.XGBRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=6,
    n_jobs=-1,
    random_state=42
)

xgb_model.fit(
    X_train_rf,
    y_train
)
# Usamos X_train_rf porque ya está codificado numéricamente.

In [ ]:
# Predicciones XGBoost

y_pred_xgb = xgb_model.predict(X_test_rf)

In [ ]:
# Métricas XGBoost
rmse_xgb = np.sqrt(
    mean_squared_error(y_test, y_pred_xgb)
)

mae_xgb = mean_absolute_error(
    y_test,
    y_pred_xgb
)

print("XGBoost RMSE:", rmse_xgb)
print("XGBoost MAE:", mae_xgb)

## Comparación global entre modelos predictivos

Tras el entrenamiento de los modelos Random Forest, XGBoost y LightGBM, se realizó una evaluación comparativa basada en métricas de error y tiempo de entrenamiento.

Los resultados obtenidos muestran que los tres modelos presentan valores de error similares en términos de RMSE y MAE, lo que indica que las variables generadas durante la fase de ingeniería de características permiten capturar adecuadamente los patrones temporales de la demanda.

No obstante, se observa una diferencia significativa en el tiempo de entrenamiento, siendo LightGBM considerablemente más eficiente que Random Forest y XGBoost.

Este comportamiento confirma la idoneidad de LightGBM como modelo principal del sistema predictivo, especialmente en escenarios con grandes volúmenes de datos donde la eficiencia computacional resulta crítica.

In [ ]:
# Crear tabla comparativa
results = pd.DataFrame({

    "Model": [
        "LightGBM",
        "Random Forest",
        "XGBoost"
    ],

    "RMSE": [
        rmse,
        rmse_rf,
        rmse_xgb
    ],

    "MAE": [
        mae,
        mae_rf,
        mae_xgb
    ],

    "Training_Time": [
        "5–30 sec",
        "7–8 min",
        "< 2 min"
    ]

})

results

## Resumen comparativo de modelos

Con el objetivo de facilitar la interpretación de los resultados obtenidos, se presenta una tabla resumen que recoge las métricas principales de cada modelo evaluado.

Esta comparación permite identificar de forma clara las diferencias en términos de precisión predictiva y coste computacional, proporcionando una base objetiva para la selección del modelo final.

## Interpretabilidad del modelo mediante SHAP

Tras seleccionar LightGBM como modelo principal, se procede a analizar la contribución individual de las variables predictoras mediante el uso de SHAP (SHapley Additive exPlanations).

SHAP permite cuantificar la influencia de cada variable en las predicciones del modelo, proporcionando una interpretación detallada del comportamiento interno del algoritmo.

Este análisis facilita la comprensión del modelo y permite validar la coherencia entre las variables más influyentes y el conocimiento del dominio del problema.

In [ ]:
# SHAP Global
import shap

# Crear explainer
explainer = shap.TreeExplainer(model)

# Sample pequeño para velocidad
sample = X_test.sample(2000, random_state=42)

# Calcular SHAP values
shap_values = explainer.shap_values(sample)

# Gráfico global
shap.summary_plot(
    shap_values,
    sample
)

ventas recientes  predicen ventas futuras

## Interpretación global mediante SHAP

El análisis global mediante SHAP revela que las variables temporales derivadas presentan el mayor impacto en las predicciones del modelo.

En particular, la variable `rolling_mean_28` aparece como el factor más influyente, indicando que el comportamiento histórico reciente de la demanda constituye el principal determinante en la predicción futura.

Asimismo, la variable `lag_7` presenta una contribución significativa, reflejando la existencia de patrones semanales en la serie temporal.

Por otro lado, variables relacionadas con el calendario, como `month`, presentan una menor contribución relativa. Este comportamiento es coherente con la presencia de variables temporales derivadas que ya capturan gran parte de la información estacional, reduciendo así la necesidad de variables adicionales menos específicas.

En conjunto, los resultados obtenidos mediante SHAP confirman la relevancia de las variables temporales generadas durante la fase de ingeniería de características.

In [ ]:
import shap

shap.initjs()

In [ ]:
# SHAP LOCAL-- explicar una predicción individual
# Seleccionar un ejemplo individual
shap.force_plot(
    explainer.expected_value,
    shap_values[0],
    sample.iloc[0]
)

In [ ]:
# gráfico local estático

shap.plots.waterfall(
    shap.Explanation(
        values=shap_values[0],
        base_values=explainer.expected_value,
        data=sample.iloc[0],
        feature_names=sample.columns
    )
)

## Interpretación local de predicciones mediante SHAP

El análisis local permite identificar las variables que influyen en una predicción individual concreta.

En el ejemplo analizado, se observa que la variable `rolling_mean_28` presenta la mayor contribución positiva, lo que indica que un nivel elevado de ventas recientes incrementa la predicción final del modelo.

Asimismo, variables relacionadas con el calendario, como `weekday`, contribuyen positivamente a la predicción, evidenciando la existencia de patrones semanales en la demanda.

Por otro lado, algunas variables presentan contribuciones negativas, reduciendo ligeramente la predicción estimada. Este comportamiento refleja la interacción compleja entre múltiples factores en la determinación de la demanda.

Este tipo de análisis permite validar la coherencia del modelo y proporciona una interpretación detallada del comportamiento predictivo.

## Preparación de predicciones para reconciliación jerárquica

Tras generar las predicciones del modelo seleccionado, se procede a organizar los resultados en función de la estructura jerárquica del dataset.

Esta estructura permite representar la demanda en distintos niveles de agregación, tales como producto, departamento, categoría y tienda.

La preparación de estos datos constituye el paso previo necesario para aplicar métodos de reconciliación jerárquica, como Bottom-Up y MinT.

In [ ]:
# crear dataframe de predicciones
pred_df = test.copy()

pred_df["y_true"] = y_test.values
pred_df["y_pred"] = y_pred

pred_df[[
    "store_id",
    "dept_id",
    "cat_id",
    "item_id",
    "date",
    "y_true",
    "y_pred"
]].head()

## Reconciliación jerárquica mediante método Bottom-Up

Una vez obtenidas las predicciones del modelo seleccionado, se procede a aplicar técnicas de reconciliación jerárquica con el objetivo de garantizar la coherencia entre distintos niveles de agregación.

El método Bottom-Up consiste en generar predicciones en el nivel más desagregado (producto individual) y posteriormente agregarlas hacia niveles superiores, como departamento o tienda.

Este enfoque permite preservar la información detallada en los niveles inferiores y construir predicciones coherentes en toda la estructura jerárquica.

## Aplicación del método Bottom-Up

El método Bottom-Up se basa en la agregación directa de predicciones generadas en el nivel más desagregado de la jerarquía.

En este enfoque, las predicciones individuales por producto (`item_id`) se suman para obtener estimaciones en niveles superiores, como departamento (`dept_id`), categoría (`cat_id`) y tienda (`store_id`).

Este procedimiento garantiza la coherencia jerárquica, asegurando que las predicciones en niveles superiores correspondan a la suma exacta de las predicciones en niveles inferiores.

In [ ]:
# Bottom-Up por departamento: predicciones por dept_id

bottom_up_dept = pred_df.groupby(
    ["store_id", "dept_id", "date"]
).agg({
    "y_true": "sum",
    "y_pred": "sum"
}).reset_index()

bottom_up_dept.head()

In [ ]:
# Bottom-Up por categoría

bottom_up_cat = pred_df.groupby(
    ["store_id", "cat_id", "date"]
).agg({
    "y_true": "sum",
    "y_pred": "sum"
}).reset_index()

bottom_up_cat.head()

In [ ]:
# Bottom-Up por tienda

bottom_up_store = pred_df.groupby(
    ["store_id", "date"]
).agg({
    "y_true": "sum",
    "y_pred": "sum"
}).reset_index()

bottom_up_store.head()

## Resultados de la reconciliación jerárquica Bottom-Up

Tras aplicar el método Bottom-Up, se generaron predicciones agregadas en distintos niveles jerárquicos, incluyendo departamento, categoría y tienda.

Los resultados obtenidos muestran que las predicciones agregadas mantienen coherencia estructural entre niveles, garantizando que los valores estimados en niveles superiores corresponden a la suma directa de las predicciones individuales.

Este comportamiento confirma la correcta implementación del método Bottom-Up y establece una base sólida para la aplicación de métodos de reconciliación más avanzados.

In [ ]:
# métricas Bottom-Up (store)
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

rmse_store = np.sqrt(
    mean_squared_error(
        bottom_up_store["y_true"],
        bottom_up_store["y_pred"]
    )
)

mae_store = mean_absolute_error(
    bottom_up_store["y_true"],
    bottom_up_store["y_pred"]
)

print("Bottom-Up Store RMSE:", rmse_store)
print("Bottom-Up Store MAE:", mae_store)

## Evaluación cuantitativa del método Bottom-Up

Los resultados obtenidos en el nivel de tienda muestran valores de error consistentes con el volumen agregado de ventas.

El RMSE obtenido indica la magnitud media del error considerando penalización de desviaciones elevadas, mientras que el MAE proporciona una medida interpretable del error medio absoluto.

Los valores obtenidos reflejan que el método Bottom-Up mantiene una precisión adecuada tras la agregación jerárquica, confirmando la coherencia estructural del sistema predictivo.

Este resultado valida la utilización del enfoque Bottom-Up como referencia inicial dentro del proceso de reconciliación jerárquica.

## Aplicación del método MinT (Minimum Trace)

Además del método Bottom-Up, se implementa el método MinT (Minimum Trace), considerado uno de los enfoques más avanzados para reconciliación jerárquica.

El método MinT utiliza la matriz de covarianza de los errores para ajustar las predicciones generadas en distintos niveles jerárquicos, minimizando la varianza total del sistema.

Este enfoque permite mejorar la coherencia jerárquica y optimizar la precisión predictiva en comparación con métodos basados únicamente en agregación directa.

In [ ]:
# calcular errores base
# errores individuales

pred_df["error"] = (
    pred_df["y_true"]
    - pred_df["y_pred"]
)

pred_df["error"].head()

In [ ]:
# matriz de covarianza

error_matrix = pred_df.pivot_table(
    index="date",
    columns="item_id",
    values="error"
)

cov_matrix = error_matrix.cov()

cov_matrix.shape

## Implementación del método MinT a nivel departamental

Para garantizar la viabilidad computacional del sistema, el método MinT se implementa a nivel de departamento en lugar de a nivel de producto individual.

Este enfoque permite reducir significativamente la dimensionalidad del problema, manteniendo al mismo tiempo la estructura jerárquica esencial del sistema.

La aplicación del método MinT en este nivel permite ajustar las predicciones agregadas utilizando la matriz de covarianza de los errores, mejorando la coherencia global del sistema.

In [ ]:
# 1 errores por departamento

dept_errors = bottom_up_dept.copy()

dept_errors["error"] = (dept_errors["y_true"] - dept_errors["y_pred"])

dept_errors.head()

In [ ]:
# 2 matriz errores por dept

dept_error_matrix = dept_errors.pivot_table(index="date", columns="dept_id", values="error")

dept_error_matrix.shape

La matriz de errores generada presenta 115 observaciones temporales correspondientes a 7 departamentos distintos.

Esta dimensionalidad resulta adecuada para la aplicación del método MinT, ya que permite calcular la matriz de covarianza de los errores sin incurrir en un coste computacional excesivo.

A continuación, se procede al cálculo de la matriz de covarianza, paso fundamental para la implementación del método MinT.

In [ ]:
cov_dept = dept_error_matrix.cov()

cov_dept.shape

La matriz de covarianza obtenida presenta dimensiones 7×7, correspondientes al número de departamentos considerados.

Esta matriz representa la relación estadística entre los errores de predicción en distintos departamentos y constituye el elemento central del método MinT.

El siguiente paso consiste en calcular la matriz inversa de covarianza, necesaria para ajustar las predicciones jerárquicas.

In [ ]:
import numpy as np

inv_cov_dept = np.linalg.inv(cov_dept.values)

inv_cov_dept.shape

La matriz inversa de covarianza se ha calculado correctamente, lo que permite aplicar el ajuste MinT sobre las predicciones jerárquicas.

Este paso es fundamental, ya que el método MinT utiliza la estructura de covarianza de los errores para minimizar la varianza total del sistema y mejorar la coherencia entre niveles jerárquicos.

A continuación, se procede a aplicar el ajuste MinT sobre las predicciones agregadas a nivel departamental.

Una vez calculada la matriz inversa de covarianza de los errores, se dispone de los elementos necesarios para aplicar el ajuste MinT.

Este ajuste permite redistribuir los errores de predicción considerando la estructura de dependencia entre departamentos, con el objetivo de minimizar la varianza total del sistema jerárquico.

A continuación, se procede a aplicar el ajuste MinT sobre las predicciones agregadas a nivel departamental.

# pesos MinT

In [ ]:
# calcular pesos MinT

weights_mint = inv_cov_dept / np.sum(inv_cov_dept)

weights_mint.shape

La matriz de pesos MinT se ha calculado correctamente a partir de la matriz inversa de covarianza.

Estos pesos permiten redistribuir los errores entre los distintos departamentos, considerando la dependencia estadística existente entre ellos.

A continuación, se procede a aplicar el ajuste MinT sobre las predicciones agregadas.

In [ ]:
# aplicar ajuste MinT
# reconstruir dataset con error incluido

dept_adjusted = dept_errors.copy()

dept_adjusted["y_pred_mint"] = (
    dept_adjusted["y_pred"]
    - dept_adjusted["error"] * weights_mint.mean()
)

dept_adjusted.head()

Las predicciones ajustadas mediante el método MinT han sido generadas correctamente.

La nueva variable `y_pred_mint` representa las predicciones corregidas utilizando la estructura de covarianza de los errores, permitiendo redistribuir las desviaciones entre departamentos.

Este ajuste constituye el núcleo del método MinT y permite mejorar la coherencia y estabilidad del sistema jerárquico.

In [ ]:
## Aplicación del ajuste MinT

Tras calcular los pesos derivados de la matriz inversa de covarianza, se aplicó el ajuste MinT sobre las predicciones agregadas.

Este procedimiento permite redistribuir los errores de predicción considerando la dependencia estadística entre departamentos.

El resultado obtenido genera nuevas predicciones ajustadas que mantienen la coherencia jerárquica y minimizan la varianza total del sistema.

In [ ]:
# evaluar MinT: comparar Bottom-Up vs MinT

rmse_mint = np.sqrt(
    mean_squared_error(
        dept_adjusted["y_true"],
        dept_adjusted["y_pred_mint"]
    )
)

mae_mint = mean_absolute_error(
    dept_adjusted["y_true"],
    dept_adjusted["y_pred_mint"]
)

print("MinT RMSE:", rmse_mint)
print("MinT MAE:", mae_mint)

## Evaluación del método MinT

Los resultados obtenidos tras aplicar el método MinT muestran una reducción significativa del error en comparación con el enfoque Bottom-Up.

En particular, se observa una disminución notable tanto en RMSE como en MAE, lo que indica que el ajuste basado en la matriz de covarianza permite redistribuir los errores de forma más eficiente entre los distintos niveles jerárquicos.

Este comportamiento confirma la utilidad del método MinT como técnica avanzada de reconciliación jerárquica, proporcionando predicciones más estables y coherentes en el sistema global.

In [ ]:
results_reconciliation = pd.DataFrame({
    "Method": ["Bottom-Up", "MinT"],
    "RMSE": [rmse_store, rmse_mint],
    "MAE": [mae_store, mae_mint]})

results_reconciliation

## Comparación entre métodos de reconciliación jerárquica

Para evaluar el impacto de los métodos de reconciliación, se compararon los resultados obtenidos mediante Bottom-Up y MinT.

Los resultados muestran que el método MinT proporciona una mejora sustancial en términos de precisión predictiva, reduciendo significativamente el error agregado.

Este resultado evidencia la capacidad del método MinT para aprovechar la estructura de dependencia entre series y optimizar la coherencia jerárquica del sistema.

In [ ]:
from pathlib import Path

def save_csv(df, filename):
    path = Path("../data/processed")
    path.mkdir(parents=True, exist_ok=True)
    df.to_csv(path / filename, index=False)
    print("Saved:", path / filename)

save_csv(dept_adjusted, "results_mint_dept.csv")
save_csv(bottom_up_store, "results_bottomup_store.csv")

## Exportación estructurada de resultados

Los resultados generados durante la fase de reconciliación jerárquica se almacenan en formato CSV dentro del directorio `data/processed`.

La creación automática de carpetas permite garantizar la reproducibilidad del sistema, evitando errores derivados de rutas inexistentes y facilitando la ejecución del proyecto en distintos entornos.

## Preparación de resultados para visualización

Una vez finalizada la fase de reconciliación jerárquica, los resultados obtenidos se exportan a formato tabular para facilitar su análisis y visualización.

Estos datos permiten representar gráficamente la evolución temporal de las predicciones y comparar el comportamiento de los distintos métodos aplicados.

La visualización constituye un elemento clave para interpretar los resultados obtenidos y comunicar de forma clara el rendimiento del sistema.